# Part 2 — Group Presentation & Notebook

**Group members:** 

- Dargys Perez : `Data Engineer`
  
- Cecilia Back : `Data Analyst`

- Marta Bielow : `Accounting Manager`
  
- Jessika Rosen : `Head of Finance` – accountable for regulatory compliance and overall financial processes, not directly involved in the operational analysis meeting

## Data Quality Issues
Compliance with the 10-day rule (delivery to invoice) was the focus of the work meeting.

**The following issues were raised:**
- Why does the delivery_confirmed_date in Deliveries.csv (originating from the logistics system/TMS) contain multiple date formats?
- What is the current situation regarding the time gap between delivery_confirmed_date and invoice_date? 
- Based on available historical data, how well are we complying with the 10-day rule today?
 

In [ ]:
# using pandas for data analysis
import pandas as pd

In [ ]:
# load the relevant data for the 10 days rule analysis
invoice_line = pd.read_csv("invoice_line.csv")
deliveries = pd.read_csv("deliveries.csv")


In [ ]:
# Just for checking how the "invoice_date" looks like
invoice_line['invoice_date']
# data type in the table "invoice_line"
invoice_line.dtypes
# There is a minor data quality issue here. 
# The datatipe is string and in order to be able to see the days in between we need it to be in a date format. 
# On the other hand the date in this column is standarized.

In [ ]:
# Let's do the same in the other table
# Just for checking how the "invoice_date" looks like in delivery_confirm_date
deliveries['delivery_confirm_date']
# data type in the table "deliveries"
deliveries.dtypes
# Here are more inconsistencies in the data. 
# Date format is not the same, we need to standized it and also go from string till date format.

In [ ]:
# Let's check if there are duplicates grouping by "order_id"
invoice_line.groupby("order_id")["invoice_date"].nunique().value_counts()

In [ ]:
invoice_line.groupby("order_id")["invoice_date"].value_counts()

In [ ]:
# Dropping the duplicates
invoice_order = (
    invoice_line
    .drop_duplicates(subset=["order_id"])
    [["order_id", "invoice_date"]]
)

In [ ]:
# Ensure that the data is cleaned from duplicates.
invoice_order.value_counts()

In [ ]:
#Join datasets invoice_order with deliveries to see the correlation betwenn this two dates 
# and for being able to make the mathematical operation.
df = invoice_order.merge(
    deliveries,
    on="order_id",
    how="left"
)

In [ ]:
# Convert from string to datetime
df["invoice_date"] = pd.to_datetime(df["invoice_date"])


In [ ]:
# Standarizing the date format.
df["delivery_confirm_date"] = pd.to_datetime(
    df["delivery_confirm_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [ ]:
#Calculate the date difference
df["days_diff"] = (df["invoice_date"] - df["delivery_confirm_date"]).dt.days

#Categorize the result
df["status"] = "Compliant"
df.loc[df["days_diff"] > 10, "status"] = "Late"
df.loc[df["days_diff"].isna(), "status"] = "Missing"

df["status"].value_counts()

summary = pd.DataFrame({
    "count": df["status"].value_counts(),
    "percent": (df["status"].value_counts(normalize=True) * 100).round(2)
})
 
print(summary)
 

In [ ]:
# How the "days_diff" are distributed in the dataset.

#df["days_diff"].value_counts().sort_index(ascending=False)

df["days_diff"].value_counts() \
    .sort_index(ascending=False) \
    .reset_index(name="occurrences") \
    .rename(columns={"index": "days_diff"})

In [ ]:
# Sorting out the orders with negative day_diff.
df[df["status"] == "Negative date-diff"]

In [ ]:
# Let see if there is a correlation between order_date and the negative numbers.

order_line = pd.read_csv("order_line.csv")

#Normalize order_line data to one row per order_id

order_order = (
    order_line
    .drop_duplicates(subset=["order_id"])
    [["order_id", "order_date"]]
    )

df1 = order_order.merge(
    df,
    on= "order_id",
    how="left"
)

In [ ]:
#df1[["status","order_id", "order_date", "invoice_date", "delivery_confirm_date", "delivery_country", "days_diff"]]

df1.loc[df1["status"] == "Negative date-diff",
        ["status","order_id","order_date","invoice_date",
         "delivery_confirm_date","delivery_country","days_diff"]] \
   .sort_values("days_diff")

## Data Quality Findings – Delivery vs Invoice Dates

-  The variety in date formats in Deliveries, appears to origin from the different delivery terminals used by courier companies. Since the logistic system, on the receiving end of the delivery flow, is old, it does not have the possibility to align date formats. This results in the inconsistent date representations.
-  Historical data shows that 44% of the orders comply  with the 10-day rule (0<= date_diff <=10). This indicates a significant gap to full compliance.
-  A notable share of orders (24%) show a negative date difference, where the invoice date precedes the delivery confirmation date. This may indicate differences in business processes or any inconsistency in time stamp across systems. 


















An analysis of the time difference between delivery_confirmed_date and invoice_date identified a number of cases with negative values, indicating that the invoice date precedes the delivery confirmation date.

**A subset of these cases was investigated manually in the source CSV files:**.<br>

*- Several records were confirmed to be data quality issues, primarily caused by:*.<br>
- Inconsistent or incorrect date formats.<br>
- Incorrectly recorded delivery dates.<br>
  
*- Other records could not be conclusively explained based on available data. Possible explanations include:* <br>

- Delayed registration of delivery confirmations in the logistics system (TMS).<br>
- Differences in business process (e.g. invoicing prior to delivery).<br>
- Integration or timing discrepancies between systems.<br>

*Further clarification would require:*
- Validation of business rules for invoicing vs delivery.<br>
- Review of source system definitions and event timestamps.<br>